In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    drop_last=True
)

100%|██████████| 9.91M/9.91M [00:04<00:00, 2.39MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 132kB/s]
100%|██████████| 1.65M/1.65M [00:02<00:00, 555kB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.60MB/s]


In [6]:
images, labels = next(iter(train_loader))

print("Images shape", images.shape)
print("Labels shape", labels.shape)

print("Minimum:", images.min().item())
print("Maximum:", images.max().item())

Images shape torch.Size([128, 1, 28, 28])
Labels shape torch.Size([128])
Minimum: -1.0
Maximum: 1.0


In [10]:
class DCGANGenerator(nn.Module):
    def __init__(self, noise_dim=100):
        super().__init__()

        self.model = nn.Sequential(
            # [B, 100, 1, 1] -> [B, 256, 7, 7]
            nn.ConvTranspose2d(
                noise_dim, 256,
                kernel_size=7,
                stride=1,
                padding=0,
                bias=False
            ),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            # [B, 256, 7, 7] -> [B, 128, 14, 14]
            nn.ConvTranspose2d(
                256, 128,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            # [B, 128, 14, 14] -> [B, 1, 28, 28]
            nn.ConvTranspose2d(
                128, 1,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

In [11]:
noise_dim = 100

generator = DCGANGenerator(noise_dim)

z = torch.randn(4, noise_dim, 1, 1)

fake_images = generator(z)

print("Noise shape:", z.shape)
print("Generated images shape:", fake_images.shape)

Noise shape: torch.Size([4, 100, 1, 1])
Generated images shape: torch.Size([4, 1, 28, 28])


In [12]:
class DCGANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(

            # [B, 1, 28, 28] -> [B, 64, 14, 14]
            nn.Conv2d(
                1, 64,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.LeakyReLU(0.2, inplace=True),

            # [B, 64, 14, 14] -> [B, 128, 7, 7]
            nn.Conv2d(
                64, 128,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            # [B, 128, 7, 7] -> [B, 1, 1, 1]
            nn.Conv2d(
                128, 1,
                kernel_size=7,
                stride=1,
                padding=0,
                bias=False
            )
        )
    def forward(self, x):
        x = self.model(x)
        return x.view(-1, 1)

In [13]:
discriminator = DCGANDiscriminator()

real_images, _ = next(iter(train_loader))

noise = torch.randn(4, noise_dim, 1, 1)
fake_images = generator(noise)

real_output = discriminator(real_images[:4])
fake_output = discriminator(fake_images)

print("Real images:", real_images[:4].shape)
print("Fake images:", fake_images.shape)

print("Real output:", real_output.shape)
print("Fake output:", fake_output.shape)

Real images: torch.Size([4, 1, 28, 28])
Fake images: torch.Size([4, 1, 28, 28])
Real output: torch.Size([4, 1])
Fake output: torch.Size([4, 1])


In [17]:
def weights_init(m):

    classname = m.__class__.__name__

    if classname.find("Conv") != -1:

        nn.init.normal_(m.weight.data, 0.0, 0.02)

    elif classname.find("BatchNorm") != -1:

        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [18]:
generator.apply(weights_init)
discriminator.apply(weights_init)

DCGANDiscriminator(
  (model): Sequential(
    (0): Conv2d(1, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (1): LeakyReLU(negative_slope=0.2, inplace=True)
    (2): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (4): LeakyReLU(negative_slope=0.2, inplace=True)
    (5): Conv2d(128, 1, kernel_size=(7, 7), stride=(1, 1), bias=False)
  )
)